# 002 — 自分の声で日本語を読み上げる

Kaggle の GPU で GPT-SoVITS のゼロショット音声合成を試します。自分の録音と、その録音どおりの文字起こしを参照音声として使います。最初は追加学習せず、参照音声による生成から始めます。

**前提:** Kaggle の Notebook 設定で Internet を ON、Accelerator を GPU にします。`001-voice-data.ipynb` で作った `voice_mono_32k.wav` と `reference_transcript.txt` を Kaggle Input に追加してください。GPT-SoVITS の依存関係・事前学習モデルは容量が大きく、初回セットアップに時間がかかります。

公式 README: https://github.com/RVC-Boss/GPT-SoVITS/blob/main/docs/ja/README.md
公式 API: https://github.com/RVC-Boss/GPT-SoVITS/blob/main/api_v2.py

録音と生成音声は個人情報です。公開データセットにせず、Kaggle の共有設定にも注意してください。

In [ ]:
import os, sys, subprocess, pathlib, json, time
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Kaggle Notebook の Accelerator を GPU に設定してから再実行してください。')

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'GPT-SoVITS'
OUT = WORK / 'voice_clone_output'
OUT.mkdir(parents=True, exist_ok=True)

## 1. GPT-SoVITS を取得してセットアップ

公式 Linux インストーラーを使います。GPU 世代や Kaggle のランタイム構成によっては依存関係の調整が必要になる場合があります。エラーが出たら、ログ末尾を確認してください。

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RVC-Boss/GPT-SoVITS.git', str(REPO)], check=True)
else:
    print('既存 clone を使用:', REPO)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'], check=True)
os.chdir(REPO)

# 公式 README の Linux セットアップに沿って依存関係と事前学習モデルを準備
install_log = OUT / 'install.log'
with install_log.open('w') as log:
    result = subprocess.run(['bash', 'install.sh', '--device', 'CU126', '--source', 'HF'], stdout=log, stderr=subprocess.STDOUT, text=True)
if result.returncode != 0:
    print('セットアップ失敗。ログ末尾:')
    print('\n'.join(install_log.read_text(errors='replace').splitlines()[-80:]))
    raise RuntimeError(f'install.sh failed: {result.returncode}')
print('セットアップ完了。ログ:', install_log)

## 2. 参照音声と読み上げ文を指定

`AUDIO_INPUT` と `TRANSCRIPT_INPUT` を、自分の Kaggle Input のパスに変更してください。文字起こしは録音内容と一致させます。参照音声は自分の声、または利用許可のある声だけを使用してください。

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

# TODO: Kaggle の Input に表示される実際のパスへ変更
AUDIO_INPUT = Path('/kaggle/input/your-voice-data/voice_mono_32k.wav')
TRANSCRIPT_INPUT = Path('/kaggle/input/your-voice-data/reference_transcript.txt')

if not AUDIO_INPUT.is_file():
    raise FileNotFoundError(f'音声ファイルがありません: {AUDIO_INPUT}。AUDIO_INPUT を修正してください。')
if not TRANSCRIPT_INPUT.is_file():
    raise FileNotFoundError(f'文字起こしがありません: {TRANSCRIPT_INPUT}。TRANSCRIPT_INPUT を修正してください。')

prompt_text = TRANSCRIPT_INPUT.read_text(encoding='utf-8').strip()
if not prompt_text:
    raise ValueError('参照音声の文字起こしが空です。')
print('参照音声:', AUDIO_INPUT)
print('参照テキスト:', prompt_text)
display(Audio(filename=str(AUDIO_INPUT)))

# ここを読み上げたい日本語に変更
TARGET_TEXT = 'こんにちは。これは、自分の声を使った日本語音声合成のテストです。'
print('生成する文:', TARGET_TEXT)

## 3. 推論 API を起動

API が参照する事前学習済みモデルは公式インストーラーと既定の `GPT_SoVITS/configs/tts_infer.yaml` に依存します。起動に失敗した場合は、下のログ末尾を確認してください。

In [ ]:
import requests

os.chdir(REPO)
api_log_path = OUT / 'api.log'
api_log = api_log_path.open('w')
api_process = subprocess.Popen(
    [sys.executable, 'api_v2.py', '-a', '127.0.0.1', '-p', '9880', '-c', 'GPT_SoVITS/configs/tts_infer.yaml'],
    cwd=str(REPO), stdout=api_log, stderr=subprocess.STDOUT
)
API_URL = 'http://127.0.0.1:9880'
ready = False
for _ in range(180):
    if api_process.poll() is not None:
        break
    try:
        # API 起動確認。未定義 endpoint でも HTTP 応答があれば起動済み
        requests.get(API_URL, timeout=2)
        ready = True
        break
    except requests.RequestException:
        time.sleep(2)
if not ready:
    api_log.flush()
    print('\n'.join(api_log_path.read_text(errors='replace').splitlines()[-100:]))
    raise RuntimeError('API が起動しませんでした。ログと依存関係を確認してください。')
print('API 起動済み:', API_URL)


## 4. 日本語音声を生成して再生

`text_lang` と `prompt_lang` は GPT-SoVITS API の言語コード `ja` を指定します。成功すると WAV を `/kaggle/working/voice_clone_output/generated.wav` に保存します。

In [ ]:
payload = {
    'text': TARGET_TEXT,
    'text_lang': 'ja',
    'ref_audio_path': str(AUDIO_INPUT),
    'prompt_text': prompt_text,
    'prompt_lang': 'ja',
    'text_split_method': 'cut5',
    'batch_size': 1,
    'media_type': 'wav',
    'streaming_mode': False,
}
response = requests.post(f'{API_URL}/tts', json=payload, timeout=600)
if response.status_code != 200 or not response.content.startswith(b'RIFF'):
    print('HTTP:', response.status_code)
    print(response.text[:5000])
    raise RuntimeError('音声生成に失敗しました。API エラーを確認してください。')

OUTPUT_WAV = OUT / 'generated.wav'
OUTPUT_WAV.write_bytes(response.content)
print('保存先:', OUTPUT_WAV)
display(Audio(filename=str(OUTPUT_WAV)))

## 完了

- 生成音声: `/kaggle/working/voice_clone_output/generated.wav`
- Kaggle の右側 Files パネルから WAV をダウンロードできます。
- セッション終了で `/kaggle/working` の内容は失われるため、必要な音声は必ず保存してください。
- まず声質・読み間違い・不自然な間を確認します。結果が不十分な場合にのみ、追加録音やファインチューニングを検討します。

このノートブックは Kaggle 上での実行を保証するものではありません。GPT-SoVITS の変更、CUDA、メモリ、モデル取得元によりセットアップ手順が変わることがあります。